<div style="text-align: right"><sub>This notebook is distributed under the <a href="https://creativecommons.org/licenses/by-sa/3.0/" target="_blank">Attribution-ShareAlike 3.0 International (CC BY-SA 3.0) license</a>.</sub></div>
<h1>Òptica Electromagnètica (transversalitat i superposició)  <span style="font-size:10px;"><i>by <a href="https://webgrec.ub.edu/webpages/000004/ang/dmaluenda.ub.edu.html" target="_blank">David Maluenda</a></i></span></h1>

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a href="https://campusvirtual.ub.edu/my" target="_blank">
      <img src="https://github.com/dmaluenda/hands_on_machine_learning/raw/master/resources/ub_logo-01.png" width="130"/>
    </a>
  </td>
  <td>
     &#160 &#160 &#160 &#160 &#160
  </td>
  <td>   <!-- gColab -->
    <a href="https://colab.research.google.com/github/opticaUB/material_classe/blob/main/21b_Transversalitat.ipynb" target="_blank">
      <img src="https://github.com/dmaluenda/hands_on_machine_learning/raw/master/resources/colab_logo_32px.png" />
      Run in Google Colab
    </a>
  </td>
  <td>
     &#160 &#160 &#160 &#160 &#160
  </td>
  <td>   <!-- github -->
    <a href="https://github.com/opticaUB/material_classe/blob/main/21b_Transversalitat.ipynb" target="_blank">
      <img src="https://github.com/dmaluenda/hands_on_machine_learning/raw/master/resources/github_logo_32px.png" />
      View source on GitHub
    </a>
  </td>
  <td>
      &#160 &#160 &#160 &#160 &#160
  </td>
  <td>   <!-- download -->
    <a href="https://raw.githubusercontent.com/opticaUB/material_classe/main/21b_Transversalitat.ipynb"  target="_blank"
          download="21b_Transversalitat.ipynb">
      <img src="https://github.com/dmaluenda/hands_on_machine_learning/raw/master/resources/download_logo_32px.png" />
      Download notebook
      </a>
  </td>
</table>

## Són transversals les ones electromagnètiques?

A classe hem vist que, per a una **ona plana harmònica**
$\mathbf{E} = \mathbf{E}_0\cos(k\,\mathbf{s}\cdot\mathbf{r}-\omega t+\varphi)$, la llei de Gauss en el buit dona

$$\nabla\cdot\mathbf{E} = -k\,(\mathbf{s}\cdot\mathbf{E}_0)\sin(\dots) = 0 \quad\Longrightarrow\quad \mathbf{E}_0\perp\mathbf{s},$$

i, de les altres equacions de Maxwell, $\mathbf{B}=\mathbf{s}\times\mathbf{E}$ (CGS, buit). Però una ona qualsevol és una **superposició** (Fourier) de molts modes harmònics, cadascun amb la seva $k_j$ i, en general, **la seva pròpia direcció** $\mathbf{s}_j$. Què queda de la transversalitat quan els sumem?

En aquest quadern posem fins a 6 fonts d'ona plana, les encenem una a una i mirem el camp elèctric total. Ens fixarem en tres coses:

1. **El camp**: fletxes de $\mathbf{E}$ en una graella del pla (amb $|\mathbf{E}|^2$ o $B_z$ de fons).
2. **La sonda**: en un punt $\mathbf{r}_0$, la contribució $\mathbf{E}_j$ de cada mode (posades una darrere l'altra) i el camp total.
3. **La fracció longitudinal** respecte de cada $\mathbf{s}_j$: $\langle(\mathbf{E}\cdot\mathbf{s}_j)^2\rangle/\langle|\mathbf{E}|^2\rangle$. Val 0 només si el camp total és $\perp\mathbf{s}_j$ a tot arreu.

> **Simplificació 2D.** Tots els $\mathbf{k}_j$ són dins el pla del dibuix i triem la polarització també dins el pla, $\hat{\mathbf{e}}_j=\hat{\mathbf{z}}\times\mathbf{s}_j$. L'altra polarització transversal ($\hat{\mathbf{z}}$) no es veuria en el dibuix. Aleshores $\mathbf{B}$ només té component $z$.

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Rectangle, Circle, FancyArrowPatch
from IPython.display import HTML, display

# Les animacions es calculen senceres a Python i després les reprodueix el navegador (JavaScript):
#   cap parpelleig, i funciona igual a Jupyter, VS Code i Google Colab.
matplotlib.rcParams['figure.dpi'] = 80                  # suficient per a pantalla (300 fa cada fotograma ~14x més pesat)
matplotlib.rcParams['animation.frame_format'] = 'jpeg'  # fotogrames lleugers
matplotlib.rcParams['animation.embed_limit'] = 60       # MB màxims d'una animació incrustada

## 1. Un mode: l'ona plana harmònica

Treballem en unitats on $\lambda_0 = 1$ i $c = 1$, de manera que el període de referència també és $T_0=1$. Cada font $j$ emet

$$\mathbf{E}_j(\mathbf{r},t) = E_{0j}\,\hat{\mathbf{e}}_j\cos\!\left(k_j\,\mathbf{s}_j\cdot\mathbf{r}-\omega_j t+\varphi_j\right),\qquad
\mathbf{s}_j=(\cos\theta_j,\sin\theta_j),\quad \hat{\mathbf{e}}_j=\hat{\mathbf{z}}\times\mathbf{s}_j,\quad k_j=\frac{2\pi}{\lambda_j},\quad \omega_j=c\,k_j,$$

i el camp total és la suma $\mathbf{E}=\sum_j\mathbf{E}_j$. Les dues funcions següents són **tota la física** del quadern; la resta és dibuix.

In [ ]:
def ona_plana(x, y, t, E0, theta, lam, phi):
    """
    Un mode de Fourier: ona plana harmònica amb k dins el pla del dibuix.
        E_j = E0 ê cos(k s·r − ω t + φ),   ê = ẑ × s  (⊥ s),   B_j = s × E_j  (CGS, buit)
    Unitats: λ0 = 1, c = 1  ⇒  T0 = 1.   theta i phi en graus.
    Retorna (Ex, Ey, Bz).
    """
    th = np.radians(theta)
    sx, sy = np.cos(th), np.sin(th)            # s: direcció de propagació
    ex, ey = -sy, sx                           # ê = ẑ × s   ⇒  ê·s = 0
    k = 2*np.pi / lam                          # nombre d'ona
    w = k                                      # ω = c k   (c = 1)
    E = E0 * np.cos(k*(sx*x + sy*y) - w*t + np.radians(phi))
    return E*ex, E*ey, E                       # B = s × E = (s × ê) E = ẑ E


def camp_total(x, y, t, fonts, amps=None):
    """
    Superposició E = Σ_j E_j de les fonts enceses.
    `amps` (opcional) substitueix les amplituds E0 de cada font (per fer rampes).
    """
    Ex = np.zeros(np.shape(x)); Ey = np.zeros(np.shape(x)); Bz = np.zeros(np.shape(x))
    for j, f in enumerate(fonts):
        E0 = (f['E0'] if f['on'] else 0) if amps is None else amps[j]
        if E0 == 0:
            continue
        ex, ey, bz = ona_plana(x, y, t, E0, f['theta'], f['lam'], f['phi'])
        Ex += ex; Ey += ey; Bz += bz
    return Ex, Ey, Bz

## 2. Diagnòstics

* **Fracció longitudinal** respecte d'una direcció $\mathbf{s}$: $\;\langle(\mathbf{E}\cdot\mathbf{s})^2\rangle/\langle|\mathbf{E}|^2\rangle$ sobre tot el pla.
* **Divergència** del camp total, calculada **numèricament** amb diferències centrades (no la suposem nul·la: la mesurem). La comparem amb el rotacional, que no és zero, per tenir una escala.

In [ ]:
def direccio(theta):
    th = np.radians(theta)
    return np.cos(th), np.sin(th)


def fraccio_longitudinal(Ex, Ey, theta):
    """⟨(E·s)²⟩ / ⟨|E|²⟩ sobre el pla.  Val 0  ⇔  E ⊥ s a tot arreu."""
    sx, sy = direccio(theta)
    E2 = np.sum(Ex**2 + Ey**2)
    return np.sum((Ex*sx + Ey*sy)**2) / E2 if E2 > 0 else np.nan


def divergencia_i_rotacional(x, y, t, fonts, amps=None, h=1e-4):
    """∇·E i (∇×E)_z per diferències centrades de pas h."""
    Ex_xp, Ey_xp, _ = camp_total(x + h, y, t, fonts, amps)
    Ex_xm, Ey_xm, _ = camp_total(x - h, y, t, fonts, amps)
    Ex_yp, Ey_yp, _ = camp_total(x, y + h, t, fonts, amps)
    Ex_ym, Ey_ym, _ = camp_total(x, y - h, t, fonts, amps)
    dEx_dx, dEy_dx = (Ex_xp - Ex_xm) / (2*h), (Ey_xp - Ey_xm) / (2*h)
    dEx_dy, dEy_dy = (Ex_yp - Ex_ym) / (2*h), (Ey_yp - Ey_ym) / (2*h)
    return dEx_dx + dEy_dy, dEy_dx - dEx_dy

## 3. Configuracions de fonts

Cada font és un diccionari amb l'amplitud `E0`, la direcció `theta` (graus), la longitud d'ona `lam` (en $\lambda_0$), la fase `phi` (graus) i si està encesa (`on`).

| Configuració | Què mostra |
|---|---|
| `dirs` | Diferents direccions i longituds d'ona: el cas general. |
| `pols` | Mateixa $\mathbf{s}$, harmònics $\lambda_0\cdot2.5/n$ amb $n=1,3,5,\dots$ i amplituds $1/n$: una **ona quadrada** $f(vt-\mathbf{r}\cdot\mathbf{s})$. |
| `mono` | Mateixa $\lambda$ (monocromàtica) però diferents direccions. |
| `creu` | Dues ones que es creuen. |
| `estac` | Dues ones oposades, $\mathbf{s}$ i $-\mathbf{s}$: ona estacionària. |

In [ ]:
def font(theta, lam, E0, phi=0, on=False):
    return dict(theta=theta, lam=lam, E0=E0, phi=phi, on=on)

CONFIGS = {
    'dirs':  dict(nom="Diferents direccions i λ", fonts=[
                 font(20, 1.00, 1.00, 0, on=True), font(115, 0.80, 0.80, 60), font(205, 1.25, 0.65, 140),
                 font(290, 0.90, 0.55, 220), font(65, 1.10, 0.45, 300), font(165, 0.70, 0.35, 30)]),
    'pols':  dict(nom="Mateixa s, harmònics (pols quadrat)", fonts=[
                 font(20, 2.5/n, 1/n, 270, on=(n == 1)) for n in (1, 3, 5, 7, 9, 11)]),
    'mono':  dict(nom="Mateixa λ, diferents s", fonts=[
                 font(20, 1, 1.00, 0, on=True), font(85, 1, 0.85, 45), font(150, 1, 0.70, 170),
                 font(215, 1, 0.60, 250), font(265, 1, 0.50, 90), font(330, 1, 0.40, 310)]),
    'creu':  dict(nom="Dues ones que es creuen", fonts=[
                 font(30, 1, 1, on=True), font(150, 1, 1, on=True), font(270, 1, 0.7),
                 font(90, 1, 0.7), font(210, 1, 0.5), font(330, 1, 0.5)]),
    'estac': dict(nom="Ona estacionària (s i −s)", fonts=[
                 font(0, 1, 1, on=True), font(180, 1, 1, on=True), font(90, 1, 0.8),
                 font(270, 1, 0.8), font(45, 1, 0.6), font(225, 1, 0.6)]),
}

def config(nom, totes=False):
    """Còpia de les fonts d'una configuració (opcionalment, totes enceses)."""
    fonts = [dict(f) for f in CONFIGS[nom]['fonts']]
    if totes:
        for f in fonts:
            f['on'] = True
    return fonts

Abans d'animar res, una comprovació numèrica amb **totes les fonts enceses**: la fracció longitudinal respecte de cada $\mathbf{s}_j$ i la divergència.

In [ ]:
L = 5.0                                              # costat del camp de visió (en λ0)
_x = (np.arange(140) + 0.5) * L/140 - L/2
X, Y = np.meshgrid(_x, _x)

for nom in CONFIGS:
    fonts = config(nom, totes=True)
    Ex, Ey, _ = camp_total(X, Y, 0.3, fonts)
    fr = [fraccio_longitudinal(Ex, Ey, f['theta']) for f in fonts]
    div, rot = divergencia_i_rotacional(X[::10, ::10], Y[::10, ::10], 0.3, fonts)
    print(f"{CONFIGS[nom]['nom']:<38}  E·s_j: " + "  ".join(f"{100*v:4.0f}%" for v in fr)
          + f"   |∇·E|/|∇×E| = {np.sqrt(np.mean(div**2) / np.mean(rot**2)):.0e}")

## 4. L'animació

La funció `anima` dibuixa el camp (esquerra), la sonda (dalt a la dreta) i la fracció longitudinal (a sota). Amb `una_a_una=True` les fonts enceses s'hi van afegint progressivament, cada $1.25\,T_0$.

*Detalls del dibuix:* les fletxes estan normalitzades a $\sqrt{\sum_j E_{0j}^2}$ i centrades al punt de la graella; les fonts són els cercles numerats de la vora, amb una fletxa que indica $\mathbf{s}_j$ (buits i discontinus si estan apagats).

In [ ]:
COLORS = ['#d1343e', '#1f6fd1', '#b37b00', '#1b8a5a', '#7f4fd0', '#cc3d8a']
INK, GRID = '#162030', '#c9d0da'
M  = 0.45      # marge al voltant del camp, on hi ha les fonts (λ0)
NQ = 21        # fletxes per costat
NB = 140       # resolució del fons

_xq = (np.arange(NQ) + 0.5) * L/NQ - L/2
XQ, YQ = np.meshgrid(_xq, _xq)
_xb = (np.arange(NB) + 0.5) * L/NB - L/2
XB, YB = np.meshgrid(_xb, _xb)
_xd = (np.arange(7) + 0.5) * L/7 - L/2
XD, YD = np.meshgrid(_xd, _xd)


def _posicio_font(theta):
    """La font es dibuixa a la vora, al costat d'on ve l'ona (a −s)."""
    sx, sy = direccio(theta)
    r = (L/2) / max(abs(sx), abs(sy)) + 0.55*M
    return -sx*r, -sy*r


def _amplituds(fonts, t, una_a_una, Te=1.25, Tr=0.5):
    """Amplitud efectiva de cada font a l'instant t (rampa de durada Tr cada Te)."""
    a = np.zeros(len(fonts))
    actives = [j for j, f in enumerate(fonts) if f['on']]
    for n, j in enumerate(actives):
        rampa = 1 if (not una_a_una or n == 0) else np.clip((t - n*Te) / Tr, 0, 1)
        a[j] = fonts[j]['E0'] * rampa
    return a


def _escala_fletxes(Ex, Ey, A, cel):
    """Longitud de fletxa ∝ |E| (saturada), normalitzada a A = sqrt(Σ E0²)."""
    mag = np.hypot(Ex, Ey)
    lon = np.minimum(1.15, mag / (1.2*A + 1e-12)) * cel * 0.9
    safe = np.where(mag > 0, mag, 1)
    return Ex/safe*lon, Ey/safe*lon


def construeix(fonts, una_a_una=False, fons='E2', periodes=2.5, sonda=(0.7, -0.5), fps=10, fotogrames_per_T0=12):
    """Prepara la figura i la funció d'actualització. Retorna (fig, update, n_fotogrames)."""
    Te = 1.25
    n_act = sum(f['on'] for f in fonts)
    durada = (n_act + 0.7) * Te if una_a_una else periodes
    nfr = max(2, int(round(durada * fotogrames_per_T0)))
    dt = durada / nfr

    fig = plt.figure(figsize=(11, 6.2))
    gs = fig.add_gridspec(2, 2, width_ratios=[1.7, 1], height_ratios=[1.3, 1],
                          left=0.01, right=0.92, top=0.91, bottom=0.07, wspace=0.12, hspace=0.35)
    axF, axP, axB = fig.add_subplot(gs[:, 0]), fig.add_subplot(gs[0, 1]), fig.add_subplot(gs[1, 1])
    titol = fig.text(0.015, 0.955, '', fontsize=13, color=INK, va='center')

    # ---------------- camp ----------------
    axF.set_xlim(-L/2 - M, L/2 + M); axF.set_ylim(-L/2 - M, L/2 + M)
    axF.set_aspect('equal'); axF.axis('off')
    kw = dict(origin='lower', extent=[-L/2, L/2, -L/2, L/2], interpolation='bilinear', zorder=0)
    if fons == 'Bz':
        im = axF.imshow(np.zeros((NB, NB)), cmap='BrBG', vmin=-1, vmax=1, **kw)
    else:
        im = axF.imshow(np.zeros((NB, NB)), cmap='Blues', vmin=0, vmax=1.15, **kw)
    im.set_visible(fons != 'cap')
    axF.add_patch(Rectangle((-L/2, -L/2), L, L, fill=False, ec=GRID, lw=1, zorder=1))
    Q = axF.quiver(XQ, YQ, np.zeros_like(XQ), np.zeros_like(XQ), angles='xy', scale_units='xy', scale=1,
                   pivot='middle', color=INK, width=0.0042, headwidth=3.6, headlength=4, headaxislength=3.6, zorder=2)
    axF.plot([-L/2 + 0.15, -L/2 + 1.15], [-L/2 + 0.18]*2, color=INK, lw=2, zorder=3)
    axF.text(-L/2 + 0.65, -L/2 + 0.24, r'$\lambda_0$', ha='center', va='bottom', color=INK, zorder=3)

    marques = []
    for j, f in enumerate(fonts):
        px, py = _posicio_font(f['theta']); sx, sy = direccio(f['theta'])
        c = Circle((px, py), 0.15, fc=COLORS[j], ec=COLORS[j], lw=1.5, zorder=5)
        ar = FancyArrowPatch((px + 0.17*sx, py + 0.17*sy), (px + 0.47*sx, py + 0.47*sy),
                             arrowstyle='-|>', mutation_scale=11, color=COLORS[j], lw=1.6, zorder=5)
        tx = axF.text(px, py, str(j + 1), ha='center', va='center', fontsize=8, weight='bold', zorder=6)
        axF.add_patch(c); axF.add_patch(ar)
        marques.append((c, ar, tx))
    axF.plot(*sonda, marker='+', ms=16, mew=1.5, color=INK, zorder=4)
    axF.plot(*sonda, marker='o', ms=10, mfc='none', mew=1.3, color=INK, zorder=4)

    # ---------------- sonda ----------------
    axP.set_xlim(-1.1, 1.1); axP.set_ylim(-1.1, 1.1); axP.set_aspect('equal'); axP.axis('off')
    axP.set_title(r'Sonda: $\mathbf{E}(\mathbf{r}_0, t)$', fontsize=11, color=INK)
    axP.add_patch(Circle((0, 0), 1, fill=False, ec=GRID, lw=1))
    axP.plot([-1, 1], [0, 0], color=GRID, lw=0.8); axP.plot([0, 0], [-1, 1], color=GRID, lw=0.8)
    linies_s = [axP.plot([], [], '--', color=COLORS[j], lw=1, alpha=0.6)[0] for j in range(len(fonts))]
    etiq_s = [axP.text(0, 0, f'$s_{j+1}$', color=COLORS[j], ha='center', va='center', fontsize=9) for j in range(len(fonts))]
    rastre, = axP.plot([], [], color=INK, lw=1, alpha=0.45)
    zeros = np.zeros(len(fonts))
    QP = axP.quiver(zeros, zeros, zeros, zeros, color=COLORS[:len(fonts)], angles='xy', scale_units='xy', scale=1,
                    width=0.013, headwidth=3.5, headlength=4, headaxislength=3.5, zorder=4)
    QT = axP.quiver([0], [0], [0], [0], color=INK, angles='xy', scale_units='xy', scale=1,
                    width=0.02, headwidth=3.2, headlength=3.8, headaxislength=3.4, zorder=5)

    # ---------------- fracció longitudinal ----------------
    ny = len(fonts)
    barres = axB.barh(range(ny), np.zeros(ny), color=COLORS[:ny], height=0.6)
    axB.set_xlim(0, 1); axB.set_ylim(ny - 0.4, -0.6)
    axB.set_yticks(range(ny), [rf'$\mathbf{{E}}\cdot\mathbf{{s}}_{j+1}$' for j in range(ny)])
    axB.set_xticks([0, 0.5, 1], ['0', '50 %', '100 %'])
    axB.set_title(r'Fracció longitudinal  $\langle(\mathbf{E}\cdot\mathbf{s}_j)^2\rangle\,/\,\langle|\mathbf{E}|^2\rangle$',
                  fontsize=11, color=INK)
    for s in ('top', 'right'):
        axB.spines[s].set_visible(False)
    valors = [axB.text(1.03, j, '', va='center', ha='left', fontsize=10, transform=axB.get_yaxis_transform())
              for j in range(ny)]
    txt_div = axB.text(0, -0.3, '', transform=axB.transAxes, fontsize=10, color=INK)

    rastre_pts = []

    def update(n):
        t = n * dt
        a = _amplituds(fonts, t, una_a_una, Te)
        A = np.sqrt(np.sum(a**2))
        actius = a > 1e-9

        # camp i fons
        Ex, Ey, Bz = camp_total(XB, YB, t, fonts, a)
        if fons == 'Bz':
            im.set_data(Bz / (1.1*A + 1e-12))
        elif fons == 'E2':
            im.set_data(np.clip((Ex**2 + Ey**2) / (1.15*A**2 + 1e-12), 0, 1)**0.8)
        Exq, Eyq, _ = camp_total(XQ, YQ, t, fonts, a)
        Q.set_UVC(*_escala_fletxes(Exq, Eyq, A, L/NQ))

        # fonts a la vora
        for j, (c, ar, tx) in enumerate(marques):
            if fonts[j]['on']:
                frac = a[j] / fonts[j]['E0'] if fonts[j]['E0'] > 0 else 0
                c.set_facecolor(COLORS[j]); c.set_linestyle('-'); c.set_alpha(0.25 + 0.75*frac)
                ar.set_visible(frac > 0); ar.set_alpha(0.25 + 0.75*frac); tx.set_color('white')
            else:
                c.set_facecolor('none'); c.set_linestyle('--'); c.set_alpha(0.7)
                ar.set_visible(False); tx.set_color(COLORS[j])

        # sonda: modes en cadena + total + rastre
        x0, y0 = sonda
        esc = 0.95 / (1.1*A + 1e-12)
        cx = cy = 0.0
        offs, U, V = [], [], []
        for j, f in enumerate(fonts):
            ex, ey, _ = ona_plana(x0, y0, t, a[j], f['theta'], f['lam'], f['phi']) if actius[j] else (0.0, 0.0, 0)
            offs.append((cx*esc, cy*esc)); U.append(ex*esc); V.append(ey*esc)
            cx += ex; cy += ey
            sx, sy = direccio(f['theta'])
            linies_s[j].set_data(([-sx, sx], [-sy, sy]) if actius[j] else ([], []))
            etiq_s[j].set_position((1.09*sx, 1.09*sy)); etiq_s[j].set_visible(bool(actius[j]))
        QP.set_offsets(offs); QP.set_UVC(U, V)
        QT.set_UVC([cx*esc], [cy*esc]); QT.set_visible(actius.sum() > 1)
        if rastre_pts and t < rastre_pts[-1][0]:
            rastre_pts.clear()                      # l'animació ha tornat a començar
        rastre_pts.append((t, cx*esc, cy*esc))
        while rastre_pts and t - rastre_pts[0][0] > 1.5:
            rastre_pts.pop(0)
        rastre.set_data([p[1] for p in rastre_pts], [p[2] for p in rastre_pts])

        # fracció longitudinal i divergència
        for j, f in enumerate(fonts):
            fr = fraccio_longitudinal(Ex, Ey, f['theta']) if actius[j] else np.nan
            barres[j].set_width(0 if np.isnan(fr) else fr)
            barres[j].set_alpha(1 if actius[j] else 0.2)
            valors[j].set_text('—' if np.isnan(fr) else ('0 ✓' if fr < 0.005 else f'{100*fr:.0f} %'))
            valors[j].set_color('#1b8a5a' if (not np.isnan(fr) and fr < 0.005) else INK)
        if actius.any():
            div, rot = divergencia_i_rotacional(XD, YD, t, fonts, a)
            r = np.sqrt(np.mean(div**2) / np.mean(rot**2))
            txt_div.set_text(rf'$\nabla\cdot\mathbf{{E}}$ = 0 ✓   ($|\nabla\cdot\mathbf{{E}}|/|\nabla\times\mathbf{{E}}|\approx$ {r:.0e}, error numèric)')
        titol.set_text(f't = {t:4.2f} T₀      ·      {int(actius.sum())} mode' + ('s' if actius.sum() != 1 else '') + ' actiu' + ('s' if actius.sum() != 1 else ''))
        return []

    return fig, update, nfr


def anima(fonts, una_a_una=False, fons='E2', periodes=2.5, sonda=(0.7, -0.5), fps=10):
    """
    fonts      llista de fonts (vegeu `config`)
    una_a_una  afegeix les fonts enceses progressivament
    fons       'E2' (|E|²), 'Bz' o 'cap'
    periodes   durada (en T0) quan una_a_una=False
    sonda      punt r0 = (x0, y0) on es mira E(r0, t)
    """
    fig, update, nfr = construeix(fonts, una_a_una, fons, periodes, sonda, fps)
    anim = FuncAnimation(fig, update, frames=nfr, interval=1000/fps)
    plt.close(fig)
    return HTML(anim.to_jshtml(default_mode='loop'))

### 4.1 Diferents direccions, afegides una a una

Primer només la font 1: les fletxes són totes $\perp\mathbf{s}_1$ i la fracció longitudinal respecte de $\mathbf{s}_1$ és 0. A mesura que s'hi afegeixen fonts amb **altres direccions**, cada mode continua sent $\perp$ a la seva $\mathbf{s}_j$ (mira la sonda), però el camp total ja no és perpendicular a cap $\mathbf{s}_j$. En canvi, $\nabla\cdot\mathbf{E}=0$ es manté.

*(Tarda uns segons a generar-se.)*

In [ ]:
anima(config('dirs', totes=True), una_a_una=True)

### 4.2 Mateixa direcció, perfil no harmònic

Ara tots els modes tenen la mateixa $\mathbf{s}$: sumen una ona quadrada $f(vt-\mathbf{r}\cdot\mathbf{s})$. El perfil ja no és harmònic, però **el camp continua sent $\perp\mathbf{s}$**. Per a qualsevol ona plana,

$$\nabla\cdot\mathbf{f}(vt-\mathbf{r}\cdot\mathbf{s}) = -\mathbf{s}\cdot\mathbf{f}' = 0 \;\Longrightarrow\; \mathbf{s}\cdot\mathbf{f} = \text{const.},$$

i la constant és un camp estàtic, no una ona. La transversalitat és una propietat de l'**ona plana** (una sola $\mathbf{s}$), no de l'harmonicitat.

In [ ]:
anima(config('pols', totes=True), una_a_una=True)

## 5. Panell interactiu

Tria una configuració, modifica les fonts i prem **Genera l'animació**. Si al teu entorn el reproductor no apareix dins el panell, executa la cel·la següent, que fa el mateix amb els valors del panell.

In [ ]:
import ipywidgets as w

NF = 6
_s = {'description_width': '52px'}
_lay = w.Layout(width='215px')

w_config = w.Dropdown(options=[(v['nom'], k) for k, v in CONFIGS.items()], value='dirs', description='Configuració',
                      style={'description_width': '90px'}, layout=w.Layout(width='360px'))
w_fons   = w.Dropdown(options=[('|E|²', 'E2'), ('B_z', 'Bz'), ('cap', 'cap')], value='E2', description='Fons',
                      layout=w.Layout(width='150px'), style={'description_width': '40px'})
w_una    = w.Checkbox(value=True, description='Afegeix-les una a una', indent=False)
w_boto   = w.Button(description="Genera l'animació", button_style='primary', icon='play', layout=w.Layout(width='190px'))
w_totes  = w.Button(description='Encén-les totes', layout=w.Layout(width='140px'))
w_sortida = w.Output()

files = []
for j in range(NF):
    fila = dict(
        on    = w.Checkbox(value=False, description='encesa', indent=False, layout=w.Layout(width='80px')),
        E0    = w.FloatSlider(1, min=0, max=1, step=0.01, description='E₀', style=_s, layout=_lay),
        theta = w.IntSlider(0, min=0, max=359, step=1, description='θ (°)', style=_s, layout=_lay),
        lam   = w.FloatSlider(1, min=0.15, max=2.5, step=0.01, description='λ/λ₀', style=_s, layout=_lay),
        phi   = w.IntSlider(0, min=0, max=359, step=1, description='φ (°)', style=_s, layout=_lay),
    )
    fila['etiqueta'] = w.HTML(f'<span style="display:inline-block;width:22px;height:22px;border-radius:50%;'
                              f'background:{COLORS[j]};color:white;text-align:center;line-height:22px;'
                              f'font-weight:bold">{j+1}</span>', layout=w.Layout(width='32px'))
    files.append(fila)


def carrega_config(change=None):
    for fila, f in zip(files, CONFIGS[w_config.value]['fonts']):
        for k in ('on', 'E0', 'theta', 'lam', 'phi'):
            fila[k].value = f[k]


def fonts_del_panell():
    return [dict(on=f['on'].value, E0=f['E0'].value, theta=f['theta'].value, lam=f['lam'].value, phi=f['phi'].value)
            for f in files]


def genera(_=None):
    with w_sortida:
        w_sortida.clear_output(wait=True)
        print('Generant l\'animació…')
        h = anima(fonts_del_panell(), una_a_una=w_una.value, fons=w_fons.value)
        w_sortida.clear_output(wait=True)
        display(h)


def encen_totes(_=None):
    for fila in files:
        fila['on'].value = True


w_config.observe(carrega_config, names='value')
w_boto.on_click(genera)
w_totes.on_click(encen_totes)
carrega_config()

display(w.VBox([
    w.HBox([w_config, w_fons, w_una]),
    *[w.HBox([f['etiqueta'], f['on'], f['E0'], f['theta'], f['lam'], f['phi']]) for f in files],
    w.HBox([w_totes, w_boto]),
    w_sortida,
]))

In [ ]:
# Alternativa sense widgets de sortida: mateixa animació amb els valors actuals del panell
anima(fonts_del_panell(), una_a_una=w_una.value, fons=w_fons.value)

## 6. Tots els casos alhora: $N = 1, 2, \dots, 6$ modes

Cada panell suma un mode més que l'anterior. A sota de cada un, $\min_j\langle(\mathbf{E}\cdot\mathbf{s}_j)^2\rangle/\langle|\mathbf{E}|^2\rangle$: si val 0, hi ha almenys una direcció $\mathbf{s}_j$ respecte de la qual el camp total és transversal.

In [ ]:
def mosaic(fonts, fons='E2', periodes=2.5, fps=10, fotogrames_per_T0=12):
    nq = 15
    xq = (np.arange(nq) + 0.5) * L/nq - L/2
    XQs, YQs = np.meshgrid(xq, xq)
    nfr = int(round(periodes * fotogrames_per_T0)); dt = periodes / nfr

    fig, axs = plt.subplots(2, 3, figsize=(11, 7.6))
    fig.subplots_adjust(left=0.01, right=0.99, top=0.95, bottom=0.03, wspace=0.04, hspace=0.18)
    panells = []
    for N, ax in enumerate(axs.flat, start=1):
        sub = [dict(f, on=(j < N)) for j, f in enumerate(fonts)]
        ax.set_xlim(-L/2 - M, L/2 + M); ax.set_ylim(-L/2 - M, L/2 + M); ax.set_aspect('equal'); ax.axis('off')
        cmap, vmin = ('BrBG', -1) if fons == 'Bz' else ('Blues', 0)
        im = ax.imshow(np.zeros((NB, NB)), origin='lower', extent=[-L/2, L/2, -L/2, L/2], cmap=cmap,
                       vmin=vmin, vmax=1 if fons == 'Bz' else 1.15, interpolation='bilinear')
        im.set_visible(fons != 'cap')
        ax.add_patch(Rectangle((-L/2, -L/2), L, L, fill=False, ec=GRID, lw=1))
        q = ax.quiver(XQs, YQs, 0*XQs, 0*XQs, angles='xy', scale_units='xy', scale=1, pivot='middle',
                      color=INK, width=0.006, headwidth=3.6, headlength=4, headaxislength=3.6)
        for j in range(N):
            px, py = _posicio_font(fonts[j]['theta']); sx, sy = direccio(fonts[j]['theta'])
            ax.add_patch(Circle((px, py), 0.17, fc=COLORS[j], ec=COLORS[j]))
            ax.add_patch(FancyArrowPatch((px + 0.2*sx, py + 0.2*sy), (px + 0.5*sx, py + 0.5*sy),
                                         arrowstyle='-|>', mutation_scale=10, color=COLORS[j], lw=1.5))
        ax.set_title(f'N = {N}', fontsize=12, color=INK)
        txt = ax.text(0, -L/2 - M - 0.05, '', ha='center', va='top', fontsize=9.5, color=INK)
        panells.append((sub, im, q, txt))

    def update(n):
        t = n * dt
        for sub, im, q, txt in panells:
            a = np.array([f['E0'] if f['on'] else 0 for f in sub]); A = np.sqrt(np.sum(a**2))
            Ex, Ey, Bz = camp_total(XB, YB, t, sub)
            if fons == 'Bz':
                im.set_data(Bz / (1.1*A))
            else:
                im.set_data(np.clip((Ex**2 + Ey**2) / (1.15*A**2), 0, 1)**0.8)
            q.set_UVC(*_escala_fletxes(*camp_total(XQs, YQs, t, sub)[:2], A, L/nq))
            fr = min(fraccio_longitudinal(Ex, Ey, f['theta']) for f in sub if f['on'])
            txt.set_text(r'$\min_j$ fracció long. = ' + ('0 ✓' if fr < 0.005 else f'{100*fr:.0f} %'))
        return []

    anim = FuncAnimation(fig, update, frames=nfr, interval=1000/fps)
    plt.close(fig)
    return HTML(anim.to_jshtml(default_mode='loop'))


mosaic(config('dirs'))

In [ ]:
# El mateix amb la configuració de mateixa s: el mínim es queda a 0 per a qualsevol N
mosaic(config('pols'))

## 7. Per pensar

1. A la configuració `estac` amb només les fonts 1 i 2 (ona estacionària, $\mathbf{s}$ i $-\mathbf{s}$) la fracció longitudinal respecte de $\mathbf{s}_1$ és 0. Per què? Hi ha ona que "viatgi" en alguna direcció?
2. La configuració `mono` és **monocromàtica** (tots els modes tenen la mateixa $\omega$) i, tot i així, el camp no és transversal a cap $\mathbf{s}_j$. Què ens diu això sobre què és realment el que garanteix $\mathbf{E}\perp\mathbf{s}$?
3. Mou la sonda a diferents punts (`anima(..., sonda=(x0, y0))`). Al llarg del temps, l'extrem de $\mathbf{E}(\mathbf{r}_0,t)$ dibuixa una línia, una el·lipse o una corba més complicada? Relaciona-ho amb l'estat de polarització.
4. Si $\nabla\cdot\mathbf{E}=0$ es compleix sempre, en quin sentit continuen sent "transversals" les ones EM? *(Pista: escriu $\mathbf{E}(\mathbf{r},t)=\int\tilde{\mathbf{E}}(\mathbf{k})\,e^{i(\mathbf{k}\cdot\mathbf{r}-\omega t)}\,d^3k$ i aplica-hi la divergència.)*